In [3]:
%pip install pandas matplotlib seaborn scipy openpyxl

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 5.0 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.8 MB 5.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.8 MB 5.4 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.8 MB 5.4 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.8 MB 5.5 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.8 MB 5.5 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 5.5 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.8 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 5.3 MB/s  0:00:01
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.5 MB 5.8 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.5 MB 5.6 MB/s eta 0:00:02
   --------------- ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

# Load your Excel dataset from the data folder
excel_path = "../data/HR_Recruitment_Funnel_Dataset (1).xlsx"
df = pd.read_excel(excel_path, sheet_name="Candidates")

print("Shape (Rows, Columns):")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Shape (Rows, Columns):
(400, 8)

Column Names:
['CandidateID', 'Department', 'JobRole', 'SourceChannel', 'ApplicationDate', 'StageReached', 'StageOrder', 'StageDate']

First 5 Rows:


,CandidateID,Department,JobRole,SourceChannel,ApplicationDate,StageReached,StageOrder,StageDate
0,C1001,Sales,Sales Representative,Company Website,2026-05-11,Offered,4,2026-06-06
1,C1002,Research & Development,Research Scientist,Job Board,2026-07-08,Interviewed,3,2026-07-22
2,C1003,Finance,Financial Analyst,Job Board,2026-05-25,Applied,1,NaT
3,C1004,Research & Development,Manufacturing Technician,LinkedIn,2026-06-26,Applied,1,NaT
4,C1005,Research & Development,Laboratory Technician,Job Board,2026-07-02,Applied,1,NaT


In [4]:
import pandas as pd

# Load your recruitment dataset
df = pd.read_excel("../data/HR_Recruitment_Funnel_Dataset (1).xlsx", sheet_name="Candidates")

# Check the dimensions (rows and columns)
print("Shape (Rows, Columns):")
print(df.shape)

# List all column names
print("\nColumn Names:")
print(df.columns.tolist())

# Preview the first 5 rows
df.head()

Shape (Rows, Columns):
(400, 8)

Column Names:
['CandidateID', 'Department', 'JobRole', 'SourceChannel', 'ApplicationDate', 'StageReached', 'StageOrder', 'StageDate']


,CandidateID,Department,JobRole,SourceChannel,ApplicationDate,StageReached,StageOrder,StageDate
0,C1001,Sales,Sales Representative,Company Website,2026-05-11,Offered,4,2026-06-06
1,C1002,Research & Development,Research Scientist,Job Board,2026-07-08,Interviewed,3,2026-07-22
2,C1003,Finance,Financial Analyst,Job Board,2026-05-25,Applied,1,NaT
3,C1004,Research & Development,Manufacturing Technician,LinkedIn,2026-06-26,Applied,1,NaT
4,C1005,Research & Development,Laboratory Technician,Job Board,2026-07-02,Applied,1,NaT


In [5]:
stages = ["Applied", "Screened", "Interviewed", "Offered", "Hired"]

for order, stage in enumerate(stages, start=1):
    reached = (df["StageOrder"] >= order).sum()
    print(stage, reached)

Applied 400
Screened 183
Interviewed 57
Offered 27
Hired 24


In [6]:
# Create a True/False column flagging final hires
df["IsHired"] = df["StageReached"] == "Hired"

# Group by sourcing channel, counting applicants and summing hires
by_channel = df.groupby("SourceChannel").agg(
    applicants=("CandidateID", "count"),
    hires=("IsHired", "sum"),
)

# Calculate hire rate percentage and round to 1 decimal place
by_channel["hire_rate_%"] = (by_channel["hires"] / by_channel["applicants"] * 100).round(1)

# Sort by highest hire rate
by_channel.sort_values("hire_rate_%", ascending=False)

,applicants,hires,hire_rate_%
SourceChannel,,,
Recruiter Sourced,41,5,12.2
Employee Referral,65,7,10.8
Company Website,61,3,4.9
LinkedIn,126,6,4.8
Job Board,107,3,2.8


In [7]:
from scipy.stats import binomtest

def wilson_ci(hires, applicants):
    """95% confidence interval for a hire rate, returned as percentages."""
    result = binomtest(hires, applicants).proportion_ci(confidence_level=0.95, method="wilson")
    return result.low * 100, result.high * 100

lows, highs = [], []
for channel in by_channel.index:
    low, high = wilson_ci(by_channel.loc[channel, "hires"], by_channel.loc[channel, "applicants"])
    lows.append(round(low, 1))
    highs.append(round(high, 1))

by_channel["ci_low_%"] = lows
by_channel["ci_high_%"] = highs
by_channel.sort_values("hire_rate_%", ascending=False)

,applicants,hires,hire_rate_%,ci_low_%,ci_high_%
SourceChannel,,,,,
Recruiter Sourced,41,5,12.2,5.3,25.5
Employee Referral,65,7,10.8,5.3,20.6
Company Website,61,3,4.9,1.7,13.5
LinkedIn,126,6,4.8,2.2,10.0
Job Board,107,3,2.8,1.0,7.9


In [8]:
from scipy.stats import fisher_exact

def compare_channels(channel_a, channel_b):
    hires_a = by_channel.loc[channel_a, "hires"]
    apps_a = by_channel.loc[channel_a, "applicants"]
    hires_b = by_channel.loc[channel_b, "hires"]
    apps_b = by_channel.loc[channel_b, "applicants"]
    
    # 2x2 contingency table: [Hired, Not Hired] for Channel A and Channel B
    table = [[hires_a, apps_a - hires_a], [hires_b, apps_b - hires_b]]
    _, p_value = fisher_exact(table)
    return round(p_value, 3)

# 1. Compare Recruiter Sourced vs Employee Referral
print("Recruiter vs Referral p-value:", compare_channels("Recruiter Sourced", "Employee Referral"))

# 2. Compare Employee Referral vs Job Board
print("Referral vs Job Board p-value:", compare_channels("Employee Referral", "Job Board"))

Recruiter vs Referral p-value: 1.0
Referral vs Job Board p-value: 0.043
